# HEP Multiagent Demo

This notebook demonstrates how to use the multi-agent framework for scientific research queries.

## Prerequisites
- Python 3.10+
- LaTeX distribution for PDF reports ([BasicTeX](https://www.tug.org/mactex/morepackages.html))
- Environment variables configured (see below)

In [ ]:
# Development mode
!pip install -e ".[dev]"

# For production:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/HEP-multiagent.git


In [ ]:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/mcp-ke.git
# mcp-ke need mcp 1.26.0 
!pip uninstall mcp_ke -y


### Set up HEP Multiagent

Set `ARGO_USER` in `.env` or pass env vars directly in server config.

In [ ]:
import os
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from hep_multiagent import Agent

load_dotenv(".env")

llm = ChatOpenAI(
    model="claudeopus46", 
    # model="claudesonnet4",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ.get("ARGO_USER", "")
)

## Initialize Agent

Create an agent with an LLM and MCP servers. Servers are auto-installed from URL at init.

In [ ]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[
    {
        # "url": "https://github.com/HEP-KE/mcp-ke.git@add-mcmc-paths",
        "url": "/Users/celsloaner/Desktop/mcp-ke/mcp-ke",
        "name": "mcp_ke"

        # for Nesar
        # "url": "/data/a/cpac/nramachandra/Projects/AmSC/mcp-ke"
    }
    ],
    lesson_memory=False,
)

for tool in agent.tools:
    print(f"  - {tool.name}")

In [ ]:
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# OUTPUT_DIR = f"./output_hepke_arxiv_demo_{timestamp}"
OUTPUT_DIR = f"./output_hepke_demo"

result = await agent.run(
    query="""
    # Run everything via MCP tools in mcp-ke. 
    # Absolutely do not use your own code or power spectra estimations (no writing new python codes, existing tools should be used). 
    # Strictly no fake/synthetic/placeholder/realistic data..

    (1) Load observational eBOSS data.
    (2) Then compare the P(k) with wCDM, ΛCDM + Massive Neutrinos and ΛCDM (use any set of parameters you need). Plot them all.
    (3) Run a full posterior analysis using MCMC? Do this for 4 parameters (sigma8, h, Σmν or sum_nu_masses, N_species) of the ΛCDM + massive neutrinos model. 
        For the target, use the observational data from eBOSS. Go with a small run. 
    (4) Show me the final posterior distribution plot via GetDist and the best-fit  estimates.
    """,
    output_dir=OUTPUT_DIR,
)